# 01 — Problem Understanding

**AttriSense · Employee Attrition Prediction & Analytics System**

This notebook documents the business problem, defines what we are predicting, and sets evaluation criteria before any data manipulation begins.

Following the MNNIT project lifecycle, problem understanding comes first. Jumping straight into EDA without this step leads to models that optimize the wrong metric for the wrong stakeholder.

## 1. Business Context

Organizations lose talent for many reasons: compensation gaps, limited growth, poor manager relationships, commute burden, or work-life imbalance. When someone leaves, the cost is not just the open headcount — it includes recruiting fees, onboarding time, and productivity loss while the role stays vacant.

HR teams typically learn about attrition risk **after** a resignation letter arrives. By then, counter-offers and retention conversations have limited effect.

**AttriSense** addresses this by:

1. **Predicting** which employees are more likely to leave (classification)
2. **Explaining** which factors contribute to that risk (analytics + SHAP, in later phases)
3. **Presenting** results in a form HR can act on (Streamlit dashboard, deployment phase)

This is a decision-support tool, not an automated termination or ranking system. Predictions should feed manager conversations and policy review, not replace human judgment.

## 2. Problem Statement

> Given an employee's demographic profile, job characteristics, satisfaction scores, and tenure history, estimate the probability that the employee will **leave the organization** within the observation period.

Formally, this is **binary classification**:

| Class | Label | Meaning |
|-------|-------|---------|
| Positive | `Yes` | Employee attrited |
| Negative | `No` | Employee stayed |

The dataset is a **snapshot**: each row is one employee, and the target reflects whether they left by the time the data was collected. We are not modeling time-to-event (that would be survival analysis — out of scope here).

## 3. Stakeholders and Decisions

| Stakeholder | Question they need answered |
|-------------|----------------------------|
| HR Business Partner | Which departments or roles show elevated attrition risk? |
| People Manager | Which team members should I prioritize for stay conversations? |
| Compensation & Benefits | Are pay and stock options correlated with departure? |
| L&D / HR Ops | Does training investment relate to retention? |

The model output is a **probability score** plus (later) **feature contributions**. HR uses the score to prioritize; managers use explanations to choose an intervention.

## 4. Success Metrics

Accuracy alone is misleading on imbalanced data. If only ~16% of employees leave, a model predicting "No" for everyone would score ~84% accuracy while being useless.

We will prioritize:

| Metric | Why it matters |
|--------|----------------|
| **Recall (attrition class)** | Missing someone who will leave is costlier than a false alarm |
| **Precision (attrition class)** | Too many false alarms erode HR trust in the system |
| **ROC-AUC** | Threshold-independent ranking quality across the workforce |
| **F1-score (attrition class)** | Balance between precision and recall for the minority class |

Business constraint: the system should **rank** employees by risk, not produce a single hard cutoff. HR can adjust the threshold based on how many intervention slots they have.

Exact metric values will be computed in the model evaluation notebook — nothing is assumed here.

## 5. Constraints and Assumptions

**Data constraints**
- Single static CSV (IBM HR Analytics Employee Attrition), ~1,470 rows
- No external HRIS integration in this project — all processing is local
- No API keys, no cloud billing, no paid services

**Modeling constraints**
- Fixed random seed (`42`) for reproducibility
- Train/test split before any feature engineering that leaks target information
- `EmployeeNumber` treated as an identifier, not a predictive feature

**Ethical considerations**
- Demographic attributes (Age, Gender, MaritalStatus) exist in the dataset. We will include them in initial modeling because they are present in the source data, but any production deployment should involve HR and legal review on fairness and permissible use.
- Predictions must not be used as sole input for adverse employment decisions.

**Out of scope**
- Real-time HRIS pipeline ingestion
- Causal inference ("will a raise reduce attrition?")
- Multi-company generalization (dataset is synthetic/single-org)

## 6. Project Lifecycle Map

The remaining notebooks follow the MNNIT lifecycle in order:

```
Problem Understanding  ← you are here
       ↓
Dataset Understanding
       ↓
Data Cleaning
       ↓
EDA
       ↓
Feature Engineering
       ↓
Model Building
       ↓
Model Evaluation
       ↓
Deployment (Streamlit)
```

Each phase produces artifacts consumed by the next. Cleaned data goes to `data/processed/`; trained models go to `models/`; plots go to `reports/figures/`.

In [ ]:
# Load project configuration to confirm paths and reproducibility settings.
# No data is touched in this notebook — only verifying the project setup.

from attrisense.config import load_config
from attrisense.utils.paths import PROJECT_ROOT, DATA_RAW_DIR

config = load_config()

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {DATA_RAW_DIR}")
print(f"Random state : {config.random_state}")
print(f"Target column: {config.target_column}")
print(f"Positive class: {config.positive_class}")